In [2]:
import pandas as pd
import os

os.makedirs('data/analytics', exist_ok=True)

sales = pd.read_csv(
    '../data/curated/sales_curated.csv'
)

customers = pd.read_csv(
    '../data/processed/customers.csv'
)

In [3]:
# Calculate customer behavior

customer_behaviour = (
    sales.groupby('customer_id')
    .agg(
        total_orders=('so_id', 'nunique'),
        total_quantity=('quantity', 'sum'),
        total_revenue=('line_grand_total', 'sum')
    )
    .reset_index()
)

customer_behaviour.to_csv(
    '../data/analytics/customer_purchase_behaviour.csv',
    index=False
)


customer_behaviour.to_csv(
    '../data/analytics/customer_purchase_behaviour.csv',
    index=False
)

In [5]:
# RFM Analysis

sales['order_date'] = pd.to_datetime(
    sales['order_date']
)

snapshot_date = (
    sales['order_date'].max()
    + pd.Timedelta(days=1)
)


rfm = (
    sales.groupby('customer_id')
    .agg({
        'order_date':
        lambda x:
        (snapshot_date - x.max()).days,

        'so_id': 'nunique',

        'line_grand_total': 'sum'
    })
)

rfm.columns = [
    'Recency',
    'Frequency',
    'Monetary'
]

rfm = rfm.reset_index()

rfm.to_csv(
    '../data/analytics/customer_rfm.csv',
    index=False
)

In [6]:
# customer segmentation using RFM analysis


rfm['segment'] = 'Bronze'

rfm.loc[
    rfm['Monetary']
    > rfm['Monetary'].quantile(.50),
    'segment'
] = 'Silver'

rfm.loc[
    rfm['Monetary']
    > rfm['Monetary'].quantile(.80),
    'segment'
] = 'Gold'

rfm.to_csv(
    '../data/analytics/customer_segments.csv',
    index=False
)

In [7]:
#Customer Lifetime Value (CLV) Analysis

customers['clv'] = (
    customers['total_purchase_value']
    /
    customers['customer_tenure_years']
)

customers[
    ['customer_id', 'clv']
].to_csv(
    '../data/analytics/customer_clv.csv',
    index=False
)

In [8]:
#customer Retention Analysis

customers['retained'] = (
    customers['days_since_last_purchase']
    <= 90
).astype(int)

customers[
    ['customer_id', 'retained']
].to_csv(
    '../data/analytics/customer_retention.csv',
    index=False
)

In [9]:
#Customer Churn Analysis

customers['churn_risk'] = (
    customers['days_since_last_purchase']
    > 180
).astype(int)

customers[
    ['customer_id',
     'churn_risk']
].to_csv(
    '../data/analytics/customer_churn.csv',
    index=False
)